<a href="https://colab.research.google.com/github/ka-stak/shakeel/blob/main/Copy_of_Netket_Jax_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1: INSTALL + CROSS-PLATFORM COMPATIBILITY LAYER
# ─────────────────────────────────────────────────────────────────────────────
# Design Principles (Audited):
#   • Fix 1 (Flax/JAX mutable_array): IN-MEMORY only → lives in Cell 2 (after
#     restart). In-memory changes are wiped by kernel restart, so the patch
#     MUST execute in the same process that imports netket. ← ARCHITECTURAL FIX
#   • Fix 2 (Python 3.13 union syntax): Single-line disk prepend located via
#     importlib.util.find_spec() — cross-platform (Colab, Windows, macOS,
#     venv, conda, CI/CD). Survives kernel restart. ← CORRECT
#   • Auto-restart guarded against CLI infinite loop via ipy is not None check.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, importlib.util, pathlib
# 1. Official pinned installation
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/netket/netket.git@v3.22.4",
     "optax>=0.1.7", "matplotlib", "scipy", "tqdm"],
    check=True
)
importlib.invalidate_caches()
# 2. FIX 2: Python 3.13 union syntax (disk patch — survives restart)
#    Uses importlib.util.find_spec → OS-agnostic path resolution
spec = importlib.util.find_spec("netket")
if spec and spec.origin:
    ops_path = pathlib.Path(spec.origin).parent / "_src" / "stats" / "online_stats" / "operations.py"
    if ops_path.exists():
        code = ops_path.read_text(encoding="utf-8")
        if "from __future__ import annotations" not in code:
            ops_path.write_text("from __future__ import annotations\n" + code, encoding="utf-8")
            print("✅ Fix 2 applied: Python 3.13 type-annotation support (operations.py)")
        else:
            print("✅ Fix 2: operations.py already clean — no changes made")
else:
    print("⚠️  Fix 2: NetKet not found via importlib — skipping disk patch")
print("=" * 65)
print("  ✅ NetKet v3.22.4 installed & configured")
print("  ♻️  Restarting kernel to load clean environment...")
print("  ⚡ Cell 2 will apply the JAX/Flax in-memory fix automatically.")
print("=" * 65)
# 3. Safe auto-restart — guarded against CLI infinite loop
#    ipy is not None only inside a live Jupyter/Colab kernel
try:
    from IPython import get_ipython
    ipy = get_ipython()
    if ipy is not None:          # ← guard: skips os.execv entirely in CLI/CI
        ipy.kernel.do_shutdown(restart=True)
except Exception:
    print("ℹ️  Manual restart required: Runtime → Restart session")


✅ Fix 2: operations.py already clean — no changes made
  ✅ NetKet v3.22.4 installed & configured
  ♻️  Restarting kernel to load clean environment...
  ⚡ Cell 2 will apply the JAX/Flax in-memory fix automatically.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2: IMPORTS & ENVIRONMENT VERIFICATION (WITH JAX 0.11.x BRIDGE)
# ─────────────────────────────────────────────────────────────────────────────
import jax
import jax._src.core as _jax_core
import jax.core as _jax_core_pub

# 1. Flax mutable_array compatibility
if not hasattr(_jax_core, "mutable_array"):
    _ArrayRef = getattr(jax, "ArrayRef", getattr(jax, "Array", None))
    if _ArrayRef is not None:
        _jax_core.mutable_array = lambda x: x   # compatibility stub (no-op)
        _jax_core.MutableArray  = _ArrayRef

# 2. JAX >= 0.11.0 compatibility: bridge symbols moved to jax.extend.core
try:
    import jax.extend.core as _jax_ext_core
    for _attr in ["get_opaque_trace_state", "get_aval"]:
        if hasattr(_jax_ext_core, _attr):
            _val = getattr(_jax_ext_core, _attr)
            setattr(_jax_core, _attr, _val)
            setattr(_jax_core_pub, _attr, _val)
except ImportError:
    pass

# ── Now safe to import NetKet ─────────────────────────────────────────────────
import jax.numpy as jnp
import netket as nk
import optax, numpy as np
import scipy.linalg as la
import warnings; warnings.filterwarnings("ignore")

print("=" * 60)
print(f"  ✅ NetKet  : {nk.__version__}")
print(f"  ✅ JAX     : {jax.__version__} (JAX 0.11 bridge active)")
print(f"  ✅ Devices : {jax.devices()}")
has_gpu = any("gpu" in str(d).lower() or "cuda" in str(d).lower() for d in jax.devices())
print("  🚀 GPU detected — ready for full training!" if has_gpu else "  ⚠️  CPU only — keep QUICK_TEST=True")
print("=" * 60)


  ✅ NetKet  : 3.22.4
  ✅ JAX     : 0.11.1 (JAX 0.11 bridge active)
  ✅ Devices : [CpuDevice(id=0)]
  ⚠️  CPU only — keep QUICK_TEST=True


In [ ]:
#────────────────────────────────────────────────────────────────────────────
# CELL 3: CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
# ┌────────────────────────────────────────────────────────────────────────┐
# │  QUICK_TEST = True  → N=6 scaled run (~10-15 min on GPU)  ← START     │
# │  QUICK_TEST = False → N=16 paper-exact run (~2-4 h on GPU)            │
# └────────────────────────────────────────────────────────────────────────┘
import numpy as np # Import numpy at the beginning of the cell
QUICK_TEST = True   # ← Set to False ONLY after N=6 validates correctly
if QUICK_TEST:
    # N=6: scaled samples for fast validation (~10-15 min on GPU)
    N              = 6
    N_SAMPLES      = 1000    # Scaled down from paper's 3000 (N=6 is smaller)
    N_SAMPLES_HARD = 2000    # Scaled down from paper's 8640
    N_CHAINS       = 16
    N_DISCARD      = 50
    N_ITER         = 500     # Scaled down from paper's 1000
    N_ITER_HARD    = 2700    # Scaled down from paper's 10000
    mode_label     = "QUICK TEST (N=6)"
else:
    # N=16: PAPER-EXACT parameters from §IV and author's code
    N              = 16
    N_SAMPLES      = 3000    # Paper: 3000 outside hard region
    N_SAMPLES_HARD = 8640    # Paper: 8640 inside hard region (1≤g/γ≤2.5)
    N_CHAINS       = 16
    N_DISCARD      = 50
    N_ITER         = 1000    # Paper: 10³ outside
    N_ITER_HARD    = 10000   # Paper: 10⁴ inside hard region
    mode_label     = "FULL PAPER (N=16)"
# ── INVIOLABLE paper parameters (PRL 2019 Eq.12, §IV) ──────────────────────
V          = 2.0       # Ising coupling: V/γ = 2
GAMMA      = 1.0       # Spontaneous decay rate γ = 1
# NDM densities (paper §III):
ALPHA_H    = 1         # Hidden density α = 1 (everywhere)
ALPHA_A    = 1         # Ancilla density β = 1 (outside hard region)
# CRITICAL (Agent 1 confirmed): In hard region, ONLY β increases, NOT α!
# Paper §IVA: α=1, β=4 for 1≤g/γ≤2.5
ALPHA_H_HARD = 1       # Hidden density α STAYS at 1 in hard region
ALPHA_A_HARD = 4       # Ancilla density β increases to 4 in hard region
# Optimizer (paper confirmed):
SR_SHIFT   = 0.01      # SR diagonal shift ε = 0.01 (paper)
LEARN_RATE = 0.02      # SGD learning rate = 0.02 (paper)
# G sweep for Fig. 2:
G_VALUES   = np.linspace(0.2, 3.0, 19)   # 19-point sweep
# Hard region flag
HARD_REGION_BOOST = True  # Paper always uses β=4 in transition region
import os
OUTPUT_DIR = "/content/NDM_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n{'─'*65}")
print(f"  Mode       : {mode_label}")
print(f"  N={N}, V/γ={V}, γ={GAMMA}")
print(f"  NDM outside hard region  : α={ALPHA_H},  β={ALPHA_A}")
print(f"  NDM in hard region       : α={ALPHA_H_HARD},  β={ALPHA_A_HARD}  (1≤g/γ≤2.5)")
print(f"  MC samples  : {N_SAMPLES} (outer) / {N_SAMPLES_HARD} (hard)")
print(f"  Steps       : {N_ITER} (outer) / {N_ITER_HARD} (hard)")
print(f"  SR_shift={SR_SHIFT}, LR={LEARN_RATE}")
print(f"  g/γ sweep   : {G_VALUES[0]:.1f} → {G_VALUES[-1]:.1f} ({len(G_VALUES)} pts)")
print(f"{'─'*65}")


─────────────────────────────────────────────────────────────────
  Mode       : QUICK TEST (N=6)
  N=6, V/γ=2.0, γ=1.0
  NDM outside hard region  : α=1,  β=1
  NDM in hard region       : α=1,  β=4  (1≤g/γ≤2.5)
  MC samples  : 1000 (outer) / 2000 (hard)
  Steps       : 500 (outer) / 2700 (hard)
  SR_shift=0.01, LR=0.02
  g/γ sweep   : 0.2 → 3.0 (19 pts)
─────────────────────────────────────────────────────────────────


In [ ]:
# CELL 4: BUILD HILBERT SPACE & OPERATORS
# ─────────────────────────────────────────────────────────────────────────────
graph   = nk.graph.Chain(length=N, pbc=True)
hilbert = nk.hilbert.Spin(s=1/2, N=N)
# Pauli matrices in {|↑⟩, |↓⟩} computational basis
SZ = np.array([[1.,  0.], [0., -1.]], dtype=complex)   # σᶻ
SX = np.array([[0.,  1.], [1.,  0.]], dtype=complex)   # σˣ
SY = np.array([[0., -1j], [1j,  0.]], dtype=complex)   # σʸ
SM = np.array([[0.,  0.], [1.,  0.]], dtype=complex)   # σ⁻ = |↓⟩⟨↑|
def build_lindbladian(hilbert, graph, g, V, gamma):
    """
    Lindblad Liouvillian for dissipative transverse-field Ising chain.
    EXACT Paper Eq.(12) — confirmed by Agent 1 from paper LaTeX source:
       H = (V/4) Σ_{<j,l>} σ̂ᶻ_j σ̂ᶻ_l  +  (g/2) Σ_j σ̂ˣ_j
       L_j = √γ σ̂⁻_j    (spontaneous emission / incoherent decay)
    V/4 coefficient CRITICAL — verified from paper Eq.(12) LaTeX.
    """
    N_sites = hilbert.size
    SZSZ    = np.kron(SZ, SZ)
    H = nk.operator.LocalOperator(hilbert, dtype=complex)
    # V/4 ZZ Ising coupling on each bond (periodic boundary)
    for i, j in graph.edges():
        H += (V / 4.0) * nk.operator.LocalOperator(hilbert, [SZSZ], [[i, j]])
    # g/2 transverse field on each site
    for i in range(N_sites):
        H += (g / 2.0) * nk.operator.LocalOperator(hilbert, [SX], [[i]])
    # Jump operators: L_j = √γ σ⁻_j
    sqrtg    = float(np.sqrt(gamma))
    jump_ops = [
        sqrtg * nk.operator.LocalOperator(hilbert, [SM], [[k]])
        for k in range(N_sites)
    ]
    return nk.operator.LocalLiouvillian(H, jump_ops)
# ── All three observables for Fig. 2 (Agent 1: paper plots σˣ, σʸ, σᶻ) ─────
Sx_total = nk.operator.LocalOperator(hilbert, dtype=complex)
Sy_total = nk.operator.LocalOperator(hilbert, dtype=complex)
Sz_total = nk.operator.LocalOperator(hilbert, dtype=complex)
for i in range(N):
    Sx_total += nk.operator.LocalOperator(hilbert, [SX], [[i]])
    Sy_total += nk.operator.LocalOperator(hilbert, [SY], [[i]])
    Sz_total += nk.operator.LocalOperator(hilbert, [SZ], [[i]])
print(f"Hilbert    : Spin-1/2, N={N}, dim={hilbert.n_states}")
print(f"Graph      : 1D Chain(pbc=True), {graph.n_edges} bonds")
print(f"Observables: ⟨σˣ⟩/N, ⟨σʸ⟩/N, ⟨σᶻ⟩/N  (all 3 components, Fig. 2)")

Hilbert    : Spin-1/2, N=6, dim=64
Graph      : 1D Chain(pbc=True), 6 bonds
Observables: ⟨σˣ⟩/N, ⟨σʸ⟩/N, ⟨σᶻ⟩/N  (all 3 components, Fig. 2)


In [ ]:
# CELL 5: PHYSICS AUDIT (7 ANALYTICAL TESTS — must all pass!)
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("  PHYSICS AUDIT — 2-site Analytical Benchmarks")
print("=" * 60)
g_test = 1.0
I2     = np.eye(2, dtype=complex)
H2_zz  = (V / 4.0) * np.kron(SZ, SZ)
H2_x   = (g_test / 2.0) * (np.kron(SX, I2) + np.kron(I2, SX))
H2     = H2_zz + H2_x
hi2   = nk.hilbert.Spin(s=1/2, N=2)
gr2   = nk.graph.Chain(length=2, pbc=True)
lind2 = build_lindbladian(hi2, gr2, g=g_test, V=V, gamma=GAMMA)
H2_nk = lind2.hamiltonian.to_dense()
# Test 1: Hamiltonian eigenvalues
eigs_manual = np.sort(np.linalg.eigvalsh(H2).real)
eigs_netket = np.sort(np.linalg.eigvalsh(H2_nk).real)
agree_H     = np.allclose(eigs_manual, eigs_netket, atol=1e-10)
print(f"\n[1] Hamiltonian eigenvalues match manual: {'✅ PASS' if agree_H else '❌ FAIL'}")
print(f"    Manual: {eigs_manual.round(4)}")
print(f"    NetKet: {eigs_netket.round(4)}")
# Test 2: V/4 coefficient (not V/2)
H2_wrong      = (V / 2.0) * np.kron(SZ, SZ) + (g_test / 2.0) * (np.kron(SX, I2) + np.kron(I2, SX))
eigs_wrong     = np.sort(np.linalg.eigvalsh(H2_wrong).real)
coeff_correct  = np.allclose(eigs_manual, eigs_netket, atol=1e-10)
coeff_wrong    = np.allclose(eigs_wrong,  eigs_netket, atol=1e-10)
print(f"\n[2] V/4 coefficient correct (not V/2): {'✅ PASS' if (coeff_correct and not coeff_wrong) else '❌ FAIL'}")
print(f"    V/4 spectrum: {eigs_manual.round(4)}")
print(f"    V/2 spectrum: {eigs_wrong.round(4)}  ← would differ")
# Test 3: σ⁻ lowering algebra
up, dn = np.array([1., 0.]), np.array([0., 1.])
test3  = np.allclose(SM @ up, dn) and np.allclose(SM @ dn, 0.)
print(f"\n[3] σ⁻ lowering operator: {'✅ PASS' if test3 else '❌ FAIL'}")
print(f"    σ⁻|↑⟩ = {SM @ up}  (expected [0. 1.])")
# Test 4: Lindblad trace preservation
dim2     = 4
rho_test = np.eye(dim2, dtype=complex) / dim2
Lop      = lind2.to_linear_operator()
Lrho     = (Lop @ rho_test.reshape(-1)).reshape(dim2, dim2)
test4    = np.abs(np.trace(Lrho)) < 1e-10
print(f"\n[4] Tr[L(ρ)] = 0 (trace-preserving): {'✅ PASS' if test4 else '❌ FAIL'}")
print(f"    |Tr[L(ρ)]| = {abs(np.trace(Lrho)):.2e}  (should be ~0)")
# Test 5: Steady state normalization
# Agent 2 confirmed: nk.exact.steady_state() returns numpy array DIRECTLY — no .to_array()
rho_ss     = nk.exact.steady_state(lind2, method="iterative")
rho_ss_arr = np.array(rho_ss)
test5      = np.abs(np.trace(rho_ss_arr).real - 1.0) < 1e-6
print(f"\n[5] Tr[ρ_ss] = 1 (normalization): {'✅ PASS' if test5 else '❌ FAIL'}")
print(f"    Tr[ρ_ss] = {np.trace(rho_ss_arr).real:.8f}")
# Test 6: Observable Hermiticity
sx_dense = Sx_total.to_dense()
is_herm  = np.allclose(sx_dense, sx_dense.conj().T, atol=1e-12)
print(f"\n[6] Observable Σσˣ is Hermitian: {'✅ PASS' if is_herm else '❌ FAIL'}")
# Test 7: Physical ordering within Fig. 2 window (g=0.2 vs g=1.5)
# NOTE: Compare g=0.2 vs g=1.5, NOT g=0.01 vs g=10 (which fails due to asymptotic mixing)
lind2_low = build_lindbladian(hi2, gr2, g=0.2, V=V, gamma=GAMMA)
lind2_mid = build_lindbladian(hi2, gr2, g=1.5, V=V, gamma=GAMMA)
ss_low    = np.array(nk.exact.steady_state(lind2_low, method="iterative"))
ss_mid    = np.array(nk.exact.steady_state(lind2_mid, method="iterative"))
sx2_op    = (nk.operator.LocalOperator(hi2, dtype=complex)
             + nk.operator.LocalOperator(hi2, [SX], [[0]])
             + nk.operator.LocalOperator(hi2, [SX], [[1]]))
sx2d      = sx2_op.to_dense()
mx_low    = abs(float(np.trace(ss_low @ sx2d).real)) / 2
mx_mid    = abs(float(np.trace(ss_mid @ sx2d).real)) / 2
test7     = mx_low < mx_mid
print(f"\n[7] ⟨σˣ⟩(g/γ=0.2) < ⟨σˣ⟩(g/γ=1.5): {'✅ PASS' if test7 else '❌ FAIL'}")
print(f"    g/γ=0.2 → {mx_low:.5f}  |  g/γ=1.5 → {mx_mid:.5f}")
all_pass = agree_H and coeff_correct and not coeff_wrong and test3 and test4 and test5 and is_herm and test7
print(f"\n{'='*60}")
print(f"  RESULT: {'✅ ALL 7 TESTS PASSED — proceed to Cell 6' if all_pass else '❌ FAILED — fix physics before continuing'}")
print(f"{'='*60}")
# %%
# ─────────────────────────────────────────────────────────────────────────────


  PHYSICS AUDIT — 2-site Analytical Benchmarks

[1] Hamiltonian eigenvalues match manual: ✅ PASS
    Manual: [-1.118 -0.5    0.5    1.118]
    NetKet: [-1.118 -0.5    0.5    1.118]

[2] V/4 coefficient correct (not V/2): ✅ PASS
    V/4 spectrum: [-1.118 -0.5    0.5    1.118]
    V/2 spectrum: [-1.4142 -1.      1.      1.4142]  ← would differ

[3] σ⁻ lowering operator: ✅ PASS
    σ⁻|↑⟩ = [0.+0.j 1.+0.j]  (expected [0. 1.])

[4] Tr[L(ρ)] = 0 (trace-preserving): ✅ PASS
    |Tr[L(ρ)]| = 0.00e+00  (should be ~0)
Starting iterative solver...
Converged trace is  (0.9999999999999968+1.5240431519713275e-15j)

[5] Tr[ρ_ss] = 1 (normalization): ✅ PASS
    Tr[ρ_ss] = 1.00000000

[6] Observable Σσˣ is Hermitian: ✅ PASS
Starting iterative solver...
Converged trace is  (0.9999999999999997-1.124166462126764e-18j)
Starting iterative solver...
Converged trace is  (0.9999999999999999+2.9872395654112135e-16j)

[7] ⟨σˣ⟩(g/γ=0.2) < ⟨σˣ⟩(g/γ=1.5): ✅ PASS
    g/γ=0.2 → 0.15485  |  g/γ=1.5 → 0.17518

  RESULT:

In [ ]:
#   CELL 6

# CELL 6: BUILD PIPELINE (CORRECTED SR SOLVER)
# ─────────────────────────────────────────────────────────────────────────────
import jax.numpy as jnp
import netket as nk
import optax

def direct_solve(A, b, **kwargs):
    """
    Direct solver replaces the default Conjugate Gradient (CG).
    Absorbs NetKet's 'x0' kwarg and returns (solution, info).
    """
    # Convert QGT object to a dense JAX array
    mat = A.to_dense() if hasattr(A, 'to_dense') else A

    # Solve exactly
    x = jnp.linalg.solve(mat, b)

    # NetKet expects a tuple of (solution, convergence_info)
    return x, None

def build_pipeline(hilbert, alpha_h, alpha_a,
                   n_samples, n_chains, n_discard, lr, sr_shift):

    # 1. Doubled Hilbert space (H⊗H)
    doubled_hilbert = nk.hilbert.DoubledHilbert(hilbert)

    # 2. NDM ansatz
    model = nk.models.NDM(alpha=alpha_h, beta=alpha_a)

    # 3. MCMC sampler on doubled space
    sampler = nk.sampler.MetropolisLocal(hilbert=doubled_hilbert, n_chains=n_chains)

    # 4. Mixed variational state
    vstate = nk.vqs.MCMixedState(
        sampler=sampler,
        model=model,
        n_samples=n_samples,
        n_discard_per_chain=n_discard,
    )
    vstate.chunk_size = 256 if not QUICK_TEST else None

    # 5. SGD
    optimizer = optax.sgd(learning_rate=lr)

    # 6. Stochastic Reconfiguration
    sr = nk.optimizer.SR(
        diag_shift=sr_shift,
        qgt=nk.optimizer.qgt.QGTJacobianDense(holomorphic=False),
        solver=direct_solve  # Uses the corrected robust solver
    )

    return vstate, optimizer, sr

# Quick validation
_vs, _, _ = build_pipeline(hilbert, ALPHA_H, ALPHA_A, N_SAMPLES, N_CHAINS, N_DISCARD, LEARN_RATE, SR_SHIFT)
n_params = _vs.n_parameters
del _vs
print(f"✅ Pipeline ready! NDM: α={ALPHA_H}, β={ALPHA_A} → {n_params} params for N={N}")

✅ Pipeline ready! NDM: α=1, β=1 → 174 params for N=6


In [ ]:
# CELL 7: EXACT BENCHMARK (all 19 g/γ points, N≤8 only)
# Agent 2: nk.exact.steady_state() returns numpy array directly — no .to_array()
# ─────────────────────────────────────────────────────────────────────────────
from tqdm.notebook import tqdm as tqdm_nb
exact_sx, exact_sy, exact_sz = [], [], []
exact_computed = False
N=6
if N <= 8:
    print(f"{'='*60}")
    print(f"  EXACT BENCHMARK — N={N} (independent computation, not from paper)")
    print(f"{'='*60}")
    sx_d = Sx_total.to_dense()
    sy_d = Sy_total.to_dense()
    sz_d = Sz_total.to_dense()
    ev_list = []
    for g_val in tqdm_nb(G_VALUES, desc="Exact diag"):
        lind_ex = build_lindbladian(hilbert, graph, g=g_val, V=V, gamma=GAMMA)
        rho_mat = np.array(nk.exact.steady_state(lind_ex, method="iterative"))
        tr      = np.trace(rho_mat).real
        ev_list.append(tr)
        exact_sx.append(float(np.trace(rho_mat @ sx_d).real) / N)
        exact_sy.append(float(np.trace(rho_mat @ sy_d).real) / N)
        exact_sz.append(float(np.trace(rho_mat @ sz_d).real) / N)
    exact_sx, exact_sy, exact_sz = np.array(exact_sx), np.array(exact_sy), np.array(exact_sz)
    exact_computed = True
    print(f"\n  All Tr[ρ] ≈ 1.0: min={min(ev_list):.6f}, max={max(ev_list):.6f}")
    print(f"\n  {'g/γ':>5}  {'⟨σˣ⟩/N':>10}  {'⟨σʸ⟩/N':>10}  {'⟨σᶻ⟩/N':>10}")
    print("  " + "─"*40)
    for gv, sx, sy, sz in zip(G_VALUES, exact_sx, exact_sy, exact_sz):
        print(f"  {gv:.2f}  {sx:>10.5f}  {sy:>10.5f}  {sz:>10.5f}")
    print(f"\n✅ Exact benchmark complete — independent from paper data")
else:
    print(f"⚠️  N={N} > 8: exact diagonalization skipped (superspace = 4^N = {4**N})")
    exact_computed = False

  EXACT BENCHMARK — N=6 (independent computation, not from paper)


Exact diag:   0%|          | 0/19 [00:00<?, ?it/s]

Starting iterative solver...
Converged trace is  (0.9999999999999565+2.2793939678436283e-14j)
Starting iterative solver...
Converged trace is  (0.9999999999999817-8.459950460202941e-15j)
Starting iterative solver...
Converged trace is  (0.999999999999961-4.443467343358075e-15j)
Starting iterative solver...
Converged trace is  (1.0000000000000002+7.640164127642088e-16j)
Starting iterative solver...
Converged trace is  (1.0000000000000338-2.497767640501356e-15j)
Starting iterative solver...
Converged trace is  (1.0000000000000255+3.4504649436056453e-15j)
Starting iterative solver...
Converged trace is  (1.000000000000035+1.497867398826203e-14j)
Starting iterative solver...
Converged trace is  (1.0000000000000009+2.3653814333385228e-15j)
Starting iterative solver...
Converged trace is  (0.999999999999994-1.9349484063014492e-15j)
Starting iterative solver...
Converged trace is  (0.9999999999999832-7.383897485823852e-15j)
Starting iterative solver...
Converged trace is  (1.0000000000000209+

In [ ]:
# CELL 8: MAIN NDM OPTIMIZATION SWEEP (FRESH RUN - NO CHECKPOINTS)
# ─────────────────────────────────────────────────────────────────────────────
import json, time, os
import numpy as np
from tqdm.notebook import tqdm
import netket as nk

# ── CLEAR CHECKPOINTS AND START FRESH ──
import shutil
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

results = []
sweep_t0 = time.time()

print("=" * 65)
print(f"  NDM OPTIMIZATION — {len(G_VALUES)} g/γ points  |  {mode_label}")
print(f"  Paper §IVA: α={ALPHA_H} everywhere | β={ALPHA_A} outer / β={ALPHA_A_HARD} hard region")
print(f"  Genuine: adiabatic init → SR minimization → measure ⟨σˣ⟩, ⟨σʸ⟩, ⟨σᶻ⟩")
print("=" * 65)

prev_parameters = None
prev_a_h = None
prev_a_a = None

for g_idx, g_val in enumerate(G_VALUES):
    # ── 1. Set Region-Specific Hyperparameters ───────────────────────────
    in_hard = HARD_REGION_BOOST and (1.0 <= g_val <= 2.5)
    a_h = ALPHA_H_HARD if in_hard else ALPHA_H
    a_a = ALPHA_A_HARD if in_hard else ALPHA_A
    n_iter = N_ITER_HARD if in_hard else N_ITER
    n_samp = N_SAMPLES_HARD if in_hard else N_SAMPLES
    reg_lbl = f"HARD (α={a_h},β={a_a})" if in_hard else f"std  (α={a_h},β={a_a})"

    print(f"\n[{g_idx+1:02d}/{len(G_VALUES)}] g/γ={g_val:.2f}  {reg_lbl}  → {n_iter} steps, {n_samp} samples")

    # ── 2. Build Fresh Liouvillian + NDM Pipeline ────────────────────────
    lind = build_lindbladian(hilbert, graph, g=g_val, V=V, gamma=GAMMA)
    vstate, optimizer, sr = build_pipeline(
        hilbert, a_h, a_a, n_samp, N_CHAINS, N_DISCARD, LEARN_RATE, SR_SHIFT
    )

    # ── 3. Adiabatic Continuation ────────────────────────────────────────
    if prev_parameters is not None and a_h == prev_a_h and a_a == prev_a_a:
        vstate.parameters = prev_parameters

    driver = nk.driver.SteadyState(
        lindbladian=lind, optimizer=optimizer,
        variational_state=vstate, preconditioner=sr
    )

    # ── 4. Optimize ──────────────────────────────────────────────────────
    t0 = time.time()
    log = nk.logging.RuntimeLog()
    driver.run(n_iter=n_iter, out=log, show_progress=True)
    elapsed = time.time() - t0

    # ── 5. Extract Loss ──────────────────────────────────────────────────
    final_loss = float("nan")
    if hasattr(log, 'data') and log.data:
        for key in ["LdagL", "Loss", "loss"]:
            if key in log.data:
                try:
                    history = log.data[key]
                    if isinstance(history, np.ndarray):
                        final_loss = float(np.real(history[-1]))
                    elif hasattr(history, 'values'):
                        vals = list(history.values()) if callable(history.values) else history.values
                        final_loss = float(np.real(vals[-1])) if vals else float("nan")
                    elif isinstance(history, (list, tuple)):
                        final_loss = float(np.real(history[-1])) if history else float("nan")
                    break
                except:
                    pass

    # ── 6. Measure Observables ───────────────────────────────────────────
    sx_stat = vstate.expect(Sx_total)
    sy_stat = vstate.expect(Sy_total)
    sz_stat = vstate.expect(Sz_total)

    sx_ndm = float(np.real(sx_stat.mean)) / N
    sy_ndm = float(np.real(sy_stat.mean)) / N
    sz_ndm = float(np.real(sz_stat.mean)) / N
    sx_err = float(np.real(sx_stat.error_of_mean)) / N

    # ── 7. Compare to Exact ──────────────────────────────────────────────
    if exact_computed:
        sx_ex = float(exact_sx[g_idx])
        err_pct = abs(sx_ndm - sx_ex) / (abs(sx_ex) + 1e-10) * 100
        quality = "✅" if err_pct < 5.0 else ("⚠️" if err_pct < 20.0 else "❌")
        ex_str = f"  | Exact={sx_ex:.4f}  Δ={err_pct:.1f}% {quality}"
    else:
        sx_ex, ex_str = None, ""

    print(f"  → ⟨σˣ⟩/N={sx_ndm:+.4f}  ⟨σʸ⟩/N={sy_ndm:+.4f}  ⟨σᶻ⟩/N={sz_ndm:+.4f} "
          f"C(v)={final_loss:.3e}  {elapsed:.0f}s{ex_str}")

    # ── 8. Save State for Next Iteration ──────────────────────────────────
    prev_parameters = vstate.parameters
    prev_a_h = a_h
    prev_a_a = a_a

    # ── 9. Record Results ────────────────────────────────────────────────
    rec = {
        'g_over_gamma': float(g_val),
        'sx_ndm': sx_ndm,
        'sy_ndm': sy_ndm,
        'sz_ndm': sz_ndm,
        'sx_err': sx_err,
        'sx_exact_all': float(exact_sx[g_idx]) if exact_computed else None,
        'sy_exact_all': float(exact_sy[g_idx]) if exact_computed else None,
        'sz_exact_all': float(exact_sz[g_idx]) if exact_computed else None,
        'final_loss': final_loss,
        'time_s': elapsed,
        'alpha_h': a_h,
        'alpha_a': a_a,
    }
    results.append(rec)

    ckpt = f"{OUTPUT_DIR}/ckpt_g{g_val:.3f}.json"
    with open(ckpt, "w") as fh:
        json.dump(rec, fh, indent=2)

total_time = time.time() - sweep_t0
print(f"\n{'─'*65}")
print(f"  ✅ Sweep complete in {total_time/60:.1f} min  ({total_time/3600:.2f} h)")
print(f"{'─'*65}")

  NDM OPTIMIZATION — 19 g/γ points  |  QUICK TEST (N=6)
  Paper §IVA: α=1 everywhere | β=1 outer / β=4 hard region
  Genuine: adiabatic init → SR minimization → measure ⟨σˣ⟩, ⟨σʸ⟩, ⟨σᶻ⟩

[01/19] g/γ=0.20  std  (α=1,β=1)  → 500 steps, 1000 samples


  0%|          | 0/500 [00:00<?, ?it/s]

  → ⟨σˣ⟩/N=+0.1147  ⟨σʸ⟩/N=+0.0327  ⟨σᶻ⟩/N=-0.9769 C(v)=nan  146s  | Exact=0.0943  Δ=21.7% ❌

[02/19] g/γ=0.36  std  (α=1,β=1)  → 500 steps, 1000 samples


  0%|          | 0/500 [00:00<?, ?it/s]

  → ⟨σˣ⟩/N=+0.1851  ⟨σʸ⟩/N=+0.0540  ⟨σᶻ⟩/N=-0.9656 C(v)=nan  125s  | Exact=0.1682  Δ=10.0% ⚠️

[03/19] g/γ=0.51  std  (α=1,β=1)  → 500 steps, 1000 samples


...

  0%|          | 0/2700 [00:00<?, ?it/s]

  → ⟨σˣ⟩/N=+0.5170  ⟨σʸ⟩/N=+0.2511  ⟨σᶻ⟩/N=-0.7067 C(v)=nan  2285s  | Exact=0.4700  Δ=10.0% ⚠️

[09/19] g/γ=1.44  HARD (α=1,β=4)  → 2700 steps, 2000 samples


  0%|          | 0/2700 [00:00<?, ?it/s]

  → ⟨σˣ⟩/N=+0.5152  ⟨σʸ⟩/N=+0.3192  ⟨σᶻ⟩/N=-0.5605 C(v)=nan  2338s  | Exact=0.4376  Δ=17.7% ⚠️

[10/19] g/γ=1.60  HARD (α=1,β=4)  → 2700 steps, 2000 samples


  0%|          | 0/2700 [00:00<?, ?it/s]

In [ ]:
# CELL 9: SAVE RESULTS TABLE
# ─────────────────────────────────────────────────────────────────────────────
import json
summary = [{k: v for k, v in r.items() if k != "loss_history"} for r in results]
out_json = f"{OUTPUT_DIR}/all_results_N{N}.json"
with open(out_json, "w") as f:
    json.dump(summary, f, indent=2)
print(f"✅ Saved: {out_json}\n")
print(f"{'g/γ':>6}  {'⟨σˣ⟩/N NDM':>12}  {'⟨σʸ⟩/N NDM':>12}  {'⟨σᶻ⟩/N NDM':>12}  {'C(v)':>11}")
print("  " + "─"*65)
for r in summary:
    print(f"  {r['g_over_gamma']:5.2f}  {r['sx_ndm']:>+12.5f}  {r['sy_ndm']:>+12.5f}  {r['sz_ndm']:>+12.5f}  {r['final_loss']:>11.3e}")


In [ ]:
# CELL 10: FIGURE 2 REPRODUCTION — All 3 components (journal quality)
# Agent 1 confirmed: paper plots ⟨σˣ⟩, ⟨σʸ⟩, ⟨σᶻ⟩ as separate panels/lines
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({"font.family": "serif", "font.size": 12,
                     "axes.labelsize": 13, "figure.dpi": 130})
g_arr  = np.array([r["g_over_gamma"] for r in results])
sx_ndm = np.array([r["sx_ndm"] for r in results])
sy_ndm = np.array([r["sy_ndm"] for r in results])
sz_ndm = np.array([r["sz_ndm"] for r in results])
sx_err = np.array([r["sx_err"] for r in results])
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
obs_data = [
    (sx_ndm, exact_sx if exact_computed else None, r"$\langle\hat{\sigma}^x\rangle/N$", "#2563EB"),
    (sy_ndm, exact_sy if exact_computed else None, r"$\langle\hat{\sigma}^y\rangle/N$", "#7C3AED"),
    (sz_ndm, exact_sz if exact_computed else None, r"$\langle\hat{\sigma}^z\rangle/N$", "#059669"),
]
for ax, (ndm_vals, exact_vals, ylabel, color) in zip(axes, obs_data):
    ax.plot(g_arr, ndm_vals, "o-", ms=6, lw=1.8, color=color,
            label=f"NDM (α={ALPHA_H}, β={ALPHA_A}/{ALPHA_A_HARD})")
    if exact_vals is not None:
        ax.plot(G_VALUES, exact_vals, "D--", ms=5, lw=1.4,
                color="#DC2626", alpha=0.85, label=f"Exact (N={N})")
    ax.axvspan(1.0, 2.5, alpha=0.07, color="orange")
    ax.axvline(1.0, ls=":", lw=0.8, color="gray", alpha=0.5)
    ax.axvline(2.5, ls=":", lw=0.8, color="gray", alpha=0.5)
    ax.set_xlabel(r"$g\,/\,\gamma$", fontsize=13)
    ax.set_ylabel(ylabel, fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, ls="--")
fig.suptitle(
    "Fig. 2 Reproduction — Vicentini et al. PRL 122, 250503 (2019)\n"
    fr"$H = \frac{{V}}{{4}}\sum\hat\sigma^z\hat\sigma^z + \frac{{g}}{{2}}\sum\hat\sigma^x$, "
    fr"$\hat L_j=\sqrt{{\gamma}}\hat\sigma^-_j$, $V/\gamma={V}$, $N={N}$",
    fontsize=10, y=1.02
)
plt.tight_layout()
fig_path = f"{OUTPUT_DIR}/fig2_reproduction_N{N}.png"
fig.savefig(fig_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"\n✅ Figure 2 (all 3 components) saved: {fig_path}")
# %%

In [ ]:
# CELL 11: CONVERGENCE PLOT
# ─────────────────────────────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(9, 5.5))
cmap = plt.cm.coolwarm
for idx, res in enumerate(results):
    c   = cmap(idx / max(len(results) - 1, 1))
    lbl = f"g/γ={res['g_over_gamma']:.1f}" if idx % 3 == 0 else None
    if res["loss_history"]:
        ax2.semilogy(res["loss_history"], color=c, lw=0.9, alpha=0.85, label=lbl)
ax2.set_xlabel("Optimization step", fontsize=12)
ax2.set_ylabel(r"$\mathcal{C}(v) = \mathrm{Tr}[\mathcal{L}^\dagger\mathcal{L}\hat\rho]/\mathrm{Tr}[\hat\rho^2]$", fontsize=11)
ax2.set_title(f"Lindblad residual convergence — all g/γ  (N={N})", fontsize=11)
ax2.legend(fontsize=7.5, ncol=4, loc="upper right")
ax2.grid(True, alpha=0.25, ls="--")
plt.tight_layout()
conv_path = f"{OUTPUT_DIR}/convergence_N{N}.png"
fig2.savefig(conv_path, dpi=160, bbox_inches="tight")
plt.show()
# %%